# Tadhkirat — Disease & Treatment Extraction Pipeline v1

## What this does:
1. Loads `plants_only_v2.csv` (649 confirmed plants)
2. For each plant reads the full paragraph from the book
3. Asks Gemini 3 Flash to extract every disease/condition mentioned
4. For each disease extracts: treatment method + expected effect
5. Modernizes classical Arabic words that modern readers would not understand
6. Saves one row per disease — same entry_id for all diseases of the same herb
7. Saves after every herb — safe to disconnect anytime

## Output columns:
- entry_id (links to plant_identification_v1)
- arabic_name
- disease_or_condition
- treatment_method
- expected_effect

## Resume behaviour:
If Colab disconnects, just run all cells again.
Already-processed herbs are skipped automatically.

## Cell 1 — Mount Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── PATHS ───────────────────────────────────────────────────────
DRIVE_FOLDER   = '/content/drive/MyDrive/tadhkirat_dawood'
PLANTS_CSV     = f'{DRIVE_FOLDER}/plants_only_v2.csv'
OUTPUT_CSV     = f'{DRIVE_FOLDER}/disease_treatment_v1.csv'
OUTPUT_EXCEL   = f'{DRIVE_FOLDER}/disease_treatment_v1.xlsx'
PROGRESS_FILE  = f'{DRIVE_FOLDER}/disease_treatment_progress_v1.txt'

# ── VERTEX AI ───────────────────────────────────────────────────
PROJECT_ID = 'project-7f940e6f-ba1d-404b-948'
LOCATION   = 'global'
MODEL_ID   = 'gemini-3-flash-preview'

print(f'Plants CSV : {"✓ Found" if os.path.exists(PLANTS_CSV) else "✗ NOT FOUND"}')
print(f'Output CSV : {OUTPUT_CSV}')

## Cell 2 — Install Dependencies

In [ ]:
!pip install google-genai pandas openpyxl -q
print('✓ Dependencies installed.')

## Cell 3 — Authenticate & Initialize Gemini

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

test = client.models.generate_content(
    model=MODEL_ID,
    contents='قل نعم فقط'
)
print(f'✓ Gemini 3 Flash ready: {test.text.strip()}')

## Cell 4 — Extraction Prompt Builder

This prompt instructs Gemini to:
1. Read the full paragraph from the book
2. Find every disease or condition mentioned
3. For each one extract the treatment method and expected effect
4. Keep everything in Arabic as written in the book
5. Only modernize genuinely archaic words modern readers would not understand
6. Return structured output with one entry per disease

In [ ]:
def build_extraction_prompt(arabic_name: str, paragraph: str) -> str:
    prompt = f"""أنت خبير في الطب العربي الإسلامي الكلاسيكي ومتخصص في استخراج المعلومات الطبية.

لديك نص من كتاب "تذكرة أولى الألباب" لداود الأنطاكي (القرن السادس عشر الميلادي).

اسم النبات: {arabic_name}

النص الكامل من الكتاب:
{paragraph}

المطلوب:
استخرج كل مرض أو حالة طبية أو عرض يذكره النص لهذا النبات.
لكل مرض، استخرج:
١. المرض أو الحالة الطبية
٢. طريقة الاستخدام أو العلاج كما يصفها الكتاب
٣. التأثير أو الفائدة المتوقعة

قواعد مهمة جداً:
- احتفظ بالنص العربي كما هو في الكتاب قدر الإمكان
- إذا كانت الكلمة أو العبارة كلاسيكية قديمة لا يفهمها القارئ الحديث، استبدلها بمرادفها العربي الحديث المفهوم
- أمثلة على التحديث: "لطوخاً" → "طلاءً", "ضماداً" → "كضماد", "سُفوفاً" → "مسحوقاً", "معجوناً" → "خلطاً"
- لا تترجم إلى الإنجليزية - كل الإجابات بالعربية
- إذا لم يذكر الكتاب طريقة استخدام محددة لمرض ما اكتب: "كما يُستعمل النبات عموماً"
- إذا لم يذكر تأثيراً محدداً اكتب: "كما ذُكر في الكتاب"
- لا تضف معلومات من خارج النص
- إذا كان النص لا يذكر أي أمراض أو استخدامات طبية اكتب: NO_DISEASES

أجب بالتنسيق التالي بالضبط، مع فاصل --- بين كل مرض:

DISEASE: [اسم المرض أو الحالة]
TREATMENT: [طريقة الاستخدام]
EFFECT: [التأثير أو الفائدة]
---
DISEASE: [اسم المرض أو الحالة]
TREATMENT: [طريقة الاستخدام]
EFFECT: [التأثير أو الفائدة]
---

لا تكتب أي نص خارج هذا التنسيق."""

    return prompt

print('✓ Prompt builder ready.')

## Cell 5 — Extraction Function

In [ ]:
import time

def extract_diseases(row: dict) -> list:
    """
    Extracts all diseases and treatments for one herb.
    Returns a list of dicts — one per disease.
    Each dict has: entry_id, arabic_name, disease, treatment, effect
    """
    entry_id     = row['entry_id']
    arabic_name  = str(row['primary_arabic_name'])
    paragraph    = str(row['full_paragraph'])

    prompt = build_extraction_prompt(arabic_name, paragraph)

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )
        text = response.text.strip()

        # Handle no diseases case
        if 'NO_DISEASES' in text:
            return [{
                'entry_id':             entry_id,
                'arabic_name':          arabic_name,
                'disease_or_condition': 'لا توجد أمراض محددة مذكورة',
                'treatment_method':     '',
                'expected_effect':      ''
            }]

        # Split by --- separator
        blocks = [b.strip() for b in text.split('---') if b.strip()]

        results = []
        for block in blocks:
            disease   = ''
            treatment = ''
            effect    = ''

            for line in block.split('\n'):
                line = line.strip()
                if line.startswith('DISEASE:'):
                    disease = line.split(':', 1)[1].strip()
                elif line.startswith('TREATMENT:'):
                    treatment = line.split(':', 1)[1].strip()
                elif line.startswith('EFFECT:'):
                    effect = line.split(':', 1)[1].strip()

            # Only add if we got at least a disease
            if disease:
                results.append({
                    'entry_id':             entry_id,
                    'arabic_name':          arabic_name,
                    'disease_or_condition': disease,
                    'treatment_method':     treatment,
                    'expected_effect':      effect
                })

        # If parsing failed return one empty row
        if not results:
            results = [{
                'entry_id':             entry_id,
                'arabic_name':          arabic_name,
                'disease_or_condition': 'خطأ في الاستخراج',
                'treatment_method':     '',
                'expected_effect':      ''
            }]

        return results

    except Exception as e:
        return [{
            'entry_id':             entry_id,
            'arabic_name':          arabic_name,
            'disease_or_condition': f'خطأ: {str(e)[:100]}',
            'treatment_method':     '',
            'expected_effect':      ''
        }]

print('✓ Extraction function ready.')

## Cell 6 — CSV Save/Load Helpers

In [ ]:
import pandas as pd

OUTPUT_COLUMNS = [
    'entry_id',
    'arabic_name',
    'disease_or_condition',
    'treatment_method',
    'expected_effect'
]

def load_output_csv() -> pd.DataFrame:
    if os.path.exists(OUTPUT_CSV):
        return pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    return pd.DataFrame(columns=OUTPUT_COLUMNS)

def save_rows(results: list):
    """Appends multiple rows at once to the output CSV."""
    if not results:
        return
    df     = load_output_csv()
    new_df = pd.DataFrame(results)
    df     = pd.concat([df, new_df], ignore_index=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

def save_progress(idx: int):
    with open(PROGRESS_FILE, 'w') as f:
        f.write(str(idx))

def load_progress() -> int:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return int(f.read().strip())
    return -1

print('✓ CSV helpers ready.')

## Cell 7 — Load Plants & Test on First Entry
Always run this before the full loop to verify quality.

In [ ]:
df_plants = pd.read_csv(PLANTS_CSV, encoding='utf-8-sig')
print(f'✓ Loaded {len(df_plants)} plants')

# Test on first entry
print('\nTesting on first entry...')
test_row = df_plants.iloc[0].to_dict()
print(f'Plant: [{test_row["primary_arabic_name"]}]')

test_results = extract_diseases(test_row)

print(f'\nExtracted {len(test_results)} disease entries:')
for i, r in enumerate(test_results, 1):
    print(f'\n  Disease {i}:')
    print(f'    المرض   : {r["disease_or_condition"]}')
    print(f'    العلاج  : {r["treatment_method"]}')
    print(f'    التأثير : {r["expected_effect"]}')

## Cell 8 — Main Extraction Loop

Processes all 649 plants.
Saves all disease rows for each herb before moving to the next.
Fully resumable — safe to disconnect anytime.

In [ ]:
last_done  = load_progress()
resume_idx = last_done + 1
total      = len(df_plants)

existing_rows = len(load_output_csv())

print(f'Total herbs           : {total}')
print(f'Already processed     : {resume_idx}')
print(f'Remaining             : {total - resume_idx}')
print(f'Rows saved so far     : {existing_rows}')
print(f'Output                : {OUTPUT_CSV}')
print('=' * 60)

total_rows_saved  = existing_rows
total_diseases    = 0

for idx in range(resume_idx, total):
    row = df_plants.iloc[idx].to_dict()

    # Extract all diseases for this herb
    results = extract_diseases(row)

    # Save all rows for this herb immediately
    save_rows(results)
    save_progress(idx)

    total_rows_saved += len(results)
    total_diseases   += len(results)

    print(
        f'[{idx+1:>3}/{total}] '
        f'[{row["primary_arabic_name"]}] '
        f'→ {len(results)} diseases extracted | '
        f'Total rows: {total_rows_saved}'
    )

    # Rate limit
    time.sleep(1.0)

print('\n' + '=' * 60)
print('EXTRACTION COMPLETE')
print(f'Total herbs processed : {total}')
print(f'Total disease rows    : {total_rows_saved}')
print(f'Avg diseases per herb : {total_rows_saved/total:.1f}')
print(f'Output                : {OUTPUT_CSV}')

## Cell 9 — Final Summary & Excel Export

In [ ]:
df_out = load_output_csv()

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Total rows            : {len(df_out)}')
print(f'Unique herbs          : {df_out["entry_id"].nunique()}')
print(f'Avg diseases per herb : {len(df_out)/df_out["entry_id"].nunique():.1f}')
print(f'Max diseases one herb : {df_out.groupby("entry_id").size().max()}')
print(f'Min diseases one herb : {df_out.groupby("entry_id").size().min()}')
print()

# Most mentioned diseases
print('Top 15 most mentioned diseases/conditions:')
top = df_out['disease_or_condition'].value_counts().head(15)
for disease, count in top.items():
    print(f'  {count:>4}x  {disease}')
print()

# Export Excel
with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

    # Sheet 1 — All entries
    df_out.to_excel(
        writer, sheet_name='All_Diseases', index=False
    )

    # Sheet 2 — Sorted by herb
    df_out.sort_values('entry_id').to_excel(
        writer, sheet_name='By_Herb', index=False
    )

    # Sheet 3 — Sorted by disease
    df_out.sort_values('disease_or_condition').to_excel(
        writer, sheet_name='By_Disease', index=False
    )

    # Sheet 4 — Herbs with most diseases (top 50)
    top_herbs = df_out.groupby('entry_id').size().nlargest(50).index
    df_out[df_out['entry_id'].isin(top_herbs)].sort_values(
        ['entry_id', 'disease_or_condition']
    ).to_excel(
        writer, sheet_name='Most_Versatile_Herbs', index=False
    )

print(f'✓ Excel saved: {OUTPUT_EXCEL}')
print(f'  Sheet 1 All_Diseases          : {len(df_out)} rows')
print(f'  Sheet 2 By_Herb               : sorted by entry_id')
print(f'  Sheet 3 By_Disease            : sorted by disease name')
print(f'  Sheet 4 Most_Versatile_Herbs  : top 50 herbs by disease count')